# Movie Recommendation Effectiveness

## Product Problem
A recommendation system is only useful if the movies it recommends are relevant to users and ultimately lead to engagement.

## Central Question
Do system recommendations align with what users expect to enjoy, and are they associated with subsequent movie consumption?

### Subquestions
1. Expectation alignment: Does the system's predicted rating align with the user's expected rating?
2. Consumption: Are recommendations associated with subsequent consumption among movies the user had not previously consumed?

In [1]:
import pandas as pd

# Load the data
belief_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/belief_data.csv')
ratings_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/user_rating_history.csv')
add_ratings_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/ratings_for_additional_users.csv')
recommendations_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/user_recommendation_history.csv')
movies_df = pd.read_csv('../data/raw/ml_belief_2024/data_release/movies.csv')

In [2]:
# Inspect the fields
print("Belief DataFrame:")
print(belief_df.info())
print("\nRatings DataFrame:")
print(ratings_df.info())
print("\nAdditional Ratings DataFrame:")
print(add_ratings_df.info())
print("\nRecommendations DataFrame:")
print(recommendations_df.info())
print("\nMovies DataFrame:")
print(movies_df.info())

Belief DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 3004084 entries, 0 to 3004083
Data columns (total 11 columns):
 #   Column               Dtype  
---  ------               -----  
 0   userId               int64  
 1   movieId              int64  
 2   isSeen               int64  
 3   watchDate            str    
 4   userElicitRating     float64
 5   userPredictRating    float64
 6   userCertainty        int64  
 7   tstamp               str    
 8   movie_idx            int64  
 9   source               int64  
 10  systemPredictRating  float64
dtypes: float64(3), int64(6), str(2)
memory usage: 252.1 MB
None

Ratings DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 2046124 entries, 0 to 2046123
Data columns (total 4 columns):
 #   Column   Dtype  
---  ------   -----  
 0   userId   int64  
 1   movieId  int64  
 2   rating   float64
 3   tstamp   str    
dtypes: float64(1), int64(2), str(1)
memory usage: 62.4 MB
None

Additional Ratings DataFrame:
<class 'pandas.DataFrame'

```mermaid
flowchart LR
    M["Movies<br/>movies.csv<br/><br/>What the movies are"]

    T["User Beliefs<br/>belief_data.csv<br/><br/>What users think"]

    R["Recommendations<br/>user_recommendation_history.csv<br/><br/>What the system recommends"]

    B["User Behavior<br/>rating histories<br/><br/>What users actually do"]

    Q["Product Analysis<br/><br/>Are recommendations aligned with user preferences?<br/><br/>Can we predict consumption?"]

    M --> T
    M --> R
    M --> B

    T --> Q
    R --> Q
    B --> Q
```

## Data Quality

In [3]:
# Check for missing entries in each DataFrame
print("=== Missing entries ===")

for name, df in {
    "Belief": belief_df,
    "Ratings": ratings_df,
    "Additional Ratings": add_ratings_df,
    "Recommendations": recommendations_df,
    "Movies": movies_df
}.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print(f"\n{name}:")
        print(missing)
    else:
        print(f"\n{name}: No missing values")

=== Missing entries ===

Belief:
watchDate    2995545
dtype: int64

Ratings:
rating    36521
dtype: int64

Additional Ratings: No missing values

Recommendations: No missing values

Movies:
genres    442
dtype: int64


Movies has 442 missing genre values. Since genre is not needed for analysis, these missing values are kept as they are. Missing ratings and watch dates require further investigation because they may affect how we identify user expectations and subsequent movie consumption.

In [4]:
# Missing watchDate could indicate that the user has not seen the movie yet.

# Check for inconsistencies between watchDate and isSeen
# Find instances where watchDate is null but isSeen is True
watchdate_null_isseen_true = belief_df[(belief_df['watchDate'].isnull()) & (belief_df['isSeen'] == True)]
print(f"Instances where watchDate is null but isSeen is True: {len(watchdate_null_isseen_true)}")

# Instances where watchDate is not null but isSeen is False
watchdate_notnull_isseen_false = belief_df[(belief_df['watchDate'].notnull()) & (belief_df['isSeen'] == False)]
print(f"Instances where watchDate is not null but isSeen is False: {len(watchdate_notnull_isseen_false)}")

Instances where watchDate is null but isSeen is True: 27
Instances where watchDate is not null but isSeen is False: 1


28 inconsistent `watchDate` records will be excluded because their `watchDate` values conflict with the `isSeen` status. The remaining missing `watchDate` values will be kept because a missing date is expected when `isSeen = 0`.

1. **Expected missing `watchDate`** → keep
2. **Inconsistent `watchDate` / `isSeen` combination** → remove

In [5]:
# Check for common users between ratings_df and add_ratings_df
# Perhaps those with missing ratings in ratings_df are present in add_ratings_df
common_users = set(ratings_df["userId"]) & set(add_ratings_df["userId"])

print(f"Common users: {len(common_users):,}")

Common users: 0


In [6]:
# Are missing ratings concentrated in specific users?
missing_ratings = ratings_df[ratings_df["rating"].isnull()]
print(f"Missing ratings by userId:\n{missing_ratings["userId"].value_counts().head()}")

Missing ratings by userId:
userId
369821    6829
235618    1522
355443    1278
286554    1093
165658    1016
Name: count, dtype: int64


In [7]:
# For userId 369821, inspect to see if they have any non-missing ratings in the ratings_df
user_369821_ratings = ratings_df[ratings_df["userId"] == 369821]
print(f"User 369821 ratings:\n{user_369821_ratings}")

User 369821 ratings:
         userId  movieId  rating               tstamp
958956   369821      231     1.0  2021-02-19 14:44:36
958957   369821     1198     4.5  2021-02-19 14:44:59
958958   369821   122918     5.0  2021-02-19 14:45:11
958959   369821   112852     5.0  2021-02-19 14:45:22
958960   369821   122914     2.0  2021-02-19 14:45:31
...         ...      ...     ...                  ...
1508185  369821      260     3.5  2023-05-09 17:18:54
1508239  369821   164179     5.0  2023-05-09 17:47:33
1508307  369821   165493     5.0  2023-05-09 19:30:01
1508310  369821   164179     NaN  2023-05-09 19:44:36
1508332  369821     5335     0.5  2023-05-09 20:09:23

[15336 rows x 4 columns]


In [8]:
# Check movieId 164179 for userId 369821 in the ratings_df
user_369821_movie_164179 = ratings_df[(ratings_df["userId"] == 369821) & (ratings_df["movieId"] == 164179)]
print(f"User 369821 ratings for movieId 164179:\n{user_369821_movie_164179}")

User 369821 ratings for movieId 164179:
         userId  movieId  rating               tstamp
958997   369821   164179     4.5  2021-02-19 15:02:18
959587   369821   164179     NaN  2021-02-20 11:24:35
959913   369821   164179     4.5  2021-02-20 15:12:34
960809   369821   164179     NaN  2021-02-21 13:04:51
961864   369821   164179     4.5  2021-02-22 13:31:41
...         ...      ...     ...                  ...
1493568  369821   164179     NaN  2023-04-26 13:32:22
1498474  369821   164179     5.0  2023-05-01 19:10:15
1502090  369821   164179     NaN  2023-05-05 12:46:54
1508239  369821   164179     5.0  2023-05-09 17:47:33
1508310  369821   164179     NaN  2023-05-09 19:44:36

[74 rows x 4 columns]


### Key Finding

- Ratings are recorded over time because users can rate the same movie more than once.
- A missing rating may reflect a rating change. For example, a user may remove an old rating and later add a new one.
- For each recommendation at time t, prior rating history can be used to identify movies the user had already interacted with before the recommendation.
- Previously interacted movies are excluded when evaluating whether a recommendation leads to a subsequent watch.

In [9]:
# Check for duplicate entries in each DataFrame
duplicate_subsets = {
    "Belief": ["userId", "movieId", "tstamp"],
    "Ratings": ["userId", "movieId", "tstamp"],
    "Additional Ratings": ["userId", "movieId", "tstamp"],
    "Recommendations": ["userId", "movieId", "tstamp"], 
    "Movies": ["movieId"]
}

for name, df in {
    "Belief": belief_df,
    "Ratings": ratings_df,
    "Additional Ratings": add_ratings_df,
    "Recommendations": recommendations_df,
    "Movies": movies_df
}.items():
    subset = duplicate_subsets[name]
    duplicates = df.duplicated(subset=subset).sum()

    if duplicates > 0:
        print(f"{name}: {duplicates} duplicate rows based on {subset}")

Ratings: 53286 duplicate rows based on ['userId', 'movieId', 'tstamp']
Recommendations: 312 duplicate rows based on ['userId', 'movieId', 'tstamp']


In [10]:
# Inspect duplicate ratings in the ratings_df
duplicate_ratings = ratings_df[
    ratings_df.duplicated(
        subset=["userId", "movieId", "tstamp"],
        keep=False
    )
].sort_values(["userId", "movieId", "tstamp"])

duplicate_ratings.head(10)

,userId,movieId,rating,tstamp
1533327,42170,101112,3.5,2023-05-28 16:37:13
1871655,42170,101112,3.5,2023-05-28 16:37:13
1342714,43715,6593,3.0,2022-11-17 13:10:12
1342715,43715,6593,3.0,2022-11-17 13:10:12
1370,43715,6711,4.5,2005-08-12 11:17:23
2042242,43715,6711,4.5,2005-08-12 11:17:23
1718,43715,56782,4.0,2010-04-24 15:37:34
1191978,43715,56782,4.0,2010-04-24 15:37:34
910982,44282,150,4.0,2020-12-05 08:44:06
910983,44282,150,4.0,2020-12-05 08:44:06


In [11]:
# Check if any duplicate entries have different ratings for the same userId, movieId, and tstamp
duplicate_ratings.groupby(
    ["userId", "movieId", "tstamp"]
)["rating"].nunique().value_counts()

rating
1    32038
2    10662
3       59
0       22
4        5
6        1
Name: count, dtype: int64

In [12]:
conflicting_ratings = duplicate_ratings[
    duplicate_ratings.groupby(
        ["userId", "movieId", "tstamp"]
    )["rating"].transform("nunique") > 1
]

conflicting_ratings.head(10)

,userId,movieId,rating,tstamp
1139413,44282,182773,3.5,2021-12-25 16:38:35
1139414,44282,182773,3.0,2021-12-25 16:38:35
919374,44282,197013,2.5,2020-12-23 11:27:39
919375,44282,197013,3.0,2020-12-23 11:27:39
866421,44282,199013,3.5,2020-09-02 12:54:35
866422,44282,199013,3.0,2020-09-02 12:54:35
1071693,44282,200332,4.0,2021-08-21 07:20:54
1071694,44282,200332,3.5,2021-08-21 07:20:54
1207806,44282,213369,2.5,2022-03-27 12:52:04
1207807,44282,213369,3.0,2022-03-27 12:52:04


Exact duplicate rating records should be removed. Records with the same user, movie, and timestamp but different ratings will be retained because timestamps are recorded in seconds and may represent separate rating actions within the same second.

In [13]:
# Inspect duplicate entries in recommendations_df
duplicate_recommendations = recommendations_df[
    recommendations_df.duplicated(
        subset=["userId", "movieId", "tstamp"],
        keep=False
    )
].sort_values(["userId", "movieId", "tstamp"])

duplicate_recommendations.head(10)

,userId,tstamp,movieId,predictedRating
974074,165742,1705876236,110,4.303458
974082,165742,1705876236,110,4.303458
974073,165742,1705876236,953,4.382074
974081,165742,1705876236,953,4.382074
974072,165742,1705876236,3147,4.346279
974080,165742,1705876236,3147,4.346279
974075,165742,1705876236,4886,4.276263
974083,165742,1705876236,4886,4.276263
974077,165742,1705876236,6377,4.260956
974085,165742,1705876236,6377,4.260956


In [14]:
duplicate_recommendations.groupby(
    ["userId", "movieId", "tstamp"]
)["predictedRating"].nunique().value_counts()

predictedRating
1    304
Name: count, dtype: int64

304 duplicate recommendation records were identified and flagged. The duplicated records had identical user, movie, timestamp, movie, and predicted-rating values, so exact duplicate rows will be excluded.

In [15]:
# Validate values in Belief DataFrame

# If isSeen = -1 (no response),
# rating and certainty fields should be -1 in this dataset

invalid_no_response = belief_df[
    (belief_df["isSeen"] == -1) &
    (
        (belief_df["userElicitRating"] != -1) |
        (belief_df["userPredictRating"] != -1) |
        (belief_df["userCertainty"] != -1)
    )
]

print(
    "Instances where isSeen = -1 but rating/certainty fields are not -1:",
    len(invalid_no_response)
)


# If isSeen = 0 (not seen),
# userPredictRating should be 0.5–5.0 in increments of 0.5
# userCertainty should be 1–5
# userElicitRating should be not applicable
# watchDate should be missing

invalid_not_seen = belief_df[
    (belief_df["isSeen"] == 0) &
    (
        (belief_df["userPredictRating"] < 0.5) |
        (belief_df["userPredictRating"] > 5.0) |
        ((belief_df["userPredictRating"] * 2) % 1 != 0) |
        (belief_df["userCertainty"] < 1) |
        (belief_df["userCertainty"] > 5) |
        (belief_df["userElicitRating"] > 0) |
        (belief_df["watchDate"].notnull())
    )
]

print(
    "Instances where isSeen = 0 but fields are inconsistent:",
    len(invalid_not_seen)
)


# If isSeen = 1 (seen),
# userElicitRating should be 0.5–5.0 in increments of 0.5
# watchDate should be present

invalid_seen = belief_df[
    (belief_df["isSeen"] == 1) &
    (
        (belief_df["userElicitRating"] < 0.5) |
        (belief_df["userElicitRating"] > 5.0) |
        ((belief_df["userElicitRating"] * 2) % 1 != 0) |
        (belief_df["watchDate"].isnull())
    )
]

print(
    "Instances where isSeen = 1 but fields are inconsistent:",
    len(invalid_seen)
)

Instances where isSeen = -1 but rating/certainty fields are not -1: 0
Instances where isSeen = 0 but fields are inconsistent: 82
Instances where isSeen = 1 but fields are inconsistent: 27


In [16]:
invalid_not_seen = belief_df[
    (belief_df["isSeen"] == 0) &
    (belief_df["userElicitRating"] > 0)
]

print(len(invalid_not_seen))

82


In [17]:
invalid_not_seen[
    ["userId", "movieId", "isSeen",
     "userElicitRating", "userPredictRating",
     "userCertainty", "watchDate", "tstamp"]
]

,userId,movieId,isSeen,userElicitRating,userPredictRating,userCertainty,watchDate,tstamp
38901,398782,110,0,3.5,3.5,3,NaN,2023-07-05 19:49:25
43183,404862,111,0,5.0,5.0,5,NaN,2023-11-29 19:50:03
146363,398823,539,0,0.5,4.0,4,NaN,2023-07-07 00:38:26
176719,398808,608,0,4.0,3.0,4,2023-07-04 00:00:00,2023-07-06 14:51:59
226665,407405,908,0,2.5,3.0,1,NaN,2024-02-05 10:44:14
...,...,...,...,...,...,...,...,...
2788270,407158,290119,0,1.0,1.0,3,NaN,2024-01-31 04:06:06
2812746,405840,290767,0,3.0,3.5,3,NaN,2023-12-29 14:57:21
2899446,406376,295519,0,1.0,1.0,4,NaN,2024-01-11 21:57:30
2909132,408984,296327,0,4.0,4.0,3,NaN,2024-03-19 02:36:01


In [18]:
invalid_seen = belief_df[
    (belief_df["isSeen"] == 1) &
    (
        (belief_df["watchDate"].isnull())
    )
]

print(len(invalid_seen))

27


In [19]:
invalid_seen[
    ["userId", "movieId", "isSeen",
     "userElicitRating", "userPredictRating",
     "userCertainty", "watchDate", "tstamp"]
]

,userId,movieId,isSeen,userElicitRating,userPredictRating,userCertainty,watchDate,tstamp
31177,372851,62,1,4.0,0.0,0,NaN,2023-11-26 23:27:42
106118,401411,356,1,5.0,0.0,0,NaN,2023-08-28 16:11:58
131367,397081,480,1,5.0,0.0,0,NaN,2023-05-15 04:24:09
137179,408118,508,1,5.0,0.0,0,NaN,2024-02-25 16:30:44
189174,410248,736,1,3.5,0.0,0,NaN,2024-04-24 20:29:08
197581,398060,778,1,1.5,0.0,0,NaN,2023-06-14 03:54:31
284984,404618,1097,1,3.5,0.0,0,NaN,2023-11-23 20:56:44
378096,394713,1221,1,2.0,0.0,0,NaN,2023-06-26 14:48:34
393543,245845,1233,1,4.0,0.0,0,NaN,2024-03-04 20:35:32
402627,89958,1240,1,4.0,0.0,0,NaN,2023-11-11 19:23:42


27 records with isSeen = 1 but missing watchDate and 82 records with isSeen = 0 but a populated userElicitRating were identified as inconsistent with the dataset's field definitions. These records will be excluded for further analysis.

In [20]:
# Function to validate ratings in both ratings_df and add_ratings_df
def validate_ratings(df, name):
    invalid_ratings = df[
        df["rating"].notna() &
        (df["rating"] != -1) &
        (
            (df["rating"] < 0.5) |
            (df["rating"] > 5.0) |
            ((df["rating"] * 2) % 1 != 0)
        )
    ]

    print(f"{name}:")
    print("Non-standard (-1) ratings:", (df["rating"] == -1).sum())
    print("Missing (NaN) ratings:", df["rating"].isna().sum())
    print("Invalid ratings:", len(invalid_ratings))
    print()

In [21]:
# Validate values in Ratings DataFrame

validate_ratings(ratings_df, "Ratings")
validate_ratings(add_ratings_df, "Additional Ratings")

Ratings:
Non-standard (-1) ratings: 213275
Missing (NaN) ratings: 36521
Invalid ratings: 0

Additional Ratings:
Non-standard (-1) ratings: 525370
Missing (NaN) ratings: 0
Invalid ratings: 0



In [22]:
ratings_df[
    ratings_df["rating"] == -1
].head(20)

,userId,movieId,rating,tstamp
3390,50108,4979,-1.0,2003-08-31 22:37:37
3835,50602,8874,-1.0,2005-06-28 14:14:16
10679,55641,1222,-1.0,2004-03-09 09:08:41
11015,55641,26662,-1.0,2005-12-22 06:37:11
14898,59059,3,-1.0,2018-09-02 14:12:16
14924,59059,158,-1.0,2018-09-02 14:12:06
14925,59059,160,-1.0,2018-09-02 14:12:12
14929,59059,173,-1.0,2018-01-21 11:52:26
14933,59059,196,-1.0,2018-09-02 14:12:20
14935,59059,208,-1.0,2016-01-31 08:37:42


In [23]:
add_ratings_df[
    add_ratings_df["rating"] == -1
].head(20)

,userId,movieId,rating,tstamp
442,327681,21,-1.0,2018-09-29 18:15:53
466,327681,339,-1.0,2018-09-29 18:15:51
496,327681,1136,-1.0,2018-09-29 18:23:50
588,327681,3740,-1.0,2018-09-29 18:23:50
814,327681,52973,-1.0,2018-09-29 18:23:18
961,327681,88163,-1.0,2018-09-29 18:23:50
1073,327681,112183,-1.0,2018-09-29 18:23:20
1218,327681,177765,-1.0,2018-09-29 18:15:47
1236,327681,183897,-1.0,2018-09-29 18:15:46
1246,327681,187541,-1.0,2018-09-29 18:15:37


The Ratings data contains 213K -1 values and 36K missing (NaN) ratings, while the Additional Ratings data contains 525K -1 values and no missing ratings. Because the meaning of -1 is not defined in the available documentation and imputing these non-rating values is unnecessary for this analysis, they are retained. No other invalid rating values were identified.

In [24]:
# Validate values in Recommendation DataFrame

invalid_predicted_ratings = recommendations_df[
    (recommendations_df["predictedRating"] < 0.0) |
    (recommendations_df["predictedRating"] > 5.0) 
]

print("Invalid predicted ratings:", len(invalid_predicted_ratings))

Invalid predicted ratings: 0
